In [1]:
! pip install -U langchain langchain-core langchain-groq "psycopg[binary]" python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json

import psycopg

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


load_dotenv(override=True)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

GRAPH_NAME = "university_graph"

# Connect to PSQL

In [3]:
conn = psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD,
)

print("Connected to PostgreSQL")

Connected to PostgreSQL


# Age Intitlization

In [4]:
with conn.cursor() as cursor:

    cursor.execute(
        "LOAD 'age';"
    )

    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

print("Apache AGE initialized")
print("AGE graph:", GRAPH_NAME)

Apache AGE initialized
AGE graph: university_graph


# Check graph exist or not

In [5]:
with conn.cursor() as cursor:

    cursor.execute(
        f"""
        SELECT *
        FROM cypher('{GRAPH_NAME}', $$
            MATCH (n)
            RETURN count(n)
        $$) AS (
            total_nodes agtype
        );
        """
    )

    row = cursor.fetchone()

print("Total nodes:", row[0])

Total nodes: 7


# Connect Groq through LangChain

In [6]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

print("Groq ready")

Groq ready


In [7]:
def normalize_agtype(value):

    if value is None:
        return None

    if isinstance(
        value,
        (list, dict, int, float, bool)
    ):
        return value

    text = str(value).strip()

    try:
        return json.loads(text)
    except Exception:
        return text.strip('"')

# Get age node schema

In [8]:
def get_age_node_schema(
    conn,
    graph_name
):

    nodes = {}

    with conn.cursor() as cursor:

        cursor.execute(
            f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH (n)

                RETURN DISTINCT
                    labels(n),
                    keys(n)
            $$) AS (
                labels agtype,
                properties agtype
            );
            """
        )

        rows = cursor.fetchall()

    for labels_value, properties_value in rows:

        labels = normalize_agtype(
            labels_value
        )

        properties = normalize_agtype(
            properties_value
        )

        if not isinstance(labels, list):
            labels = [labels]

        if not isinstance(properties, list):
            properties = [properties]

        for label in labels:

            label = str(label)

            nodes.setdefault(
                label,
                set()
            )

            nodes[label].update(
                str(prop)
                for prop in properties
            )

    return {
        label: sorted(properties)
        for label, properties
        in nodes.items()
    }

# Get Relationship Schema

In [9]:
def get_age_relationship_schema(
    conn,
    graph_name
):

    relationships = []

    with conn.cursor() as cursor:

        cursor.execute(
            f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH (a)-[r]->(b)

                RETURN DISTINCT
                    labels(a),
                    type(r),
                    labels(b)
            $$) AS (
                source_labels agtype,
                relationship agtype,
                target_labels agtype
            );
            """
        )

        rows = cursor.fetchall()

    for (
        source_value,
        relationship_value,
        target_value
    ) in rows:

        source = normalize_agtype(
            source_value
        )

        relationship = normalize_agtype(
            relationship_value
        )

        target = normalize_agtype(
            target_value
        )

        relationships.append(
            {
                "source": source,
                "relationship": relationship,
                "target": target,
            }
        )

    return relationships

In [10]:
def get_age_graph_schema(
    conn,
    graph_name
):

    return {
        "nodes":
            get_age_node_schema(
                conn,
                graph_name,
            ),

        "relationships":
            get_age_relationship_schema(
                conn,
                graph_name,
            ),
    }

In [11]:
age_schema = get_age_graph_schema(
    conn=conn,
    graph_name=GRAPH_NAME,
)

print(age_schema)

{'nodes': {'Professor': ['docling_id', 'identifier', 'research_areas'], 'University': ['docling_id', 'identifier'], 'Course': ['docling_id', 'enrolled_students', 'identifier', 'technologies'], 'Department': ['docling_id', 'identifier', 'professors'], 'Student': ['docling_id', 'identifier', 'skills']}, 'relationships': [{'source': ['Department'], 'relationship': 'HAS_HEAD', 'target': ['Professor']}, {'source': ['Department'], 'relationship': 'HAS_STUDENT', 'target': ['Student']}, {'source': ['University'], 'relationship': 'HAS_DEPARTMENT', 'target': ['Department']}, {'source': ['University'], 'relationship': 'OFFERS_COURSE', 'target': ['Course']}]}


# Convert Schema to Text for LLM

In [12]:
def build_schema_text(
    schema
):

    lines = []

    lines.append(
        "NODE LABELS AND PROPERTIES"
    )

    for label, properties in (
        schema["nodes"].items()
    ):

        lines.append(
            f"\nNode: {label}"
        )

        lines.append(
            "Properties: "
            + ", ".join(properties)
        )


    lines.append(
        "\nRELATIONSHIPS"
    )

    for relation in (
        schema["relationships"]
    ):

        source = relation["source"]
        target = relation["target"]

        if isinstance(source, list):
            source = ", ".join(source)

        if isinstance(target, list):
            target = ", ".join(target)

        relationship = (
            relation["relationship"]
        )

        lines.append(
            f"({source})"
            f"-[:{relationship}]->"
            f"({target})"
        )

    return "\n".join(lines)

In [13]:
GRAPH_SCHEMA = build_schema_text(
    age_schema
)

print(GRAPH_SCHEMA)

NODE LABELS AND PROPERTIES

Node: Professor
Properties: docling_id, identifier, research_areas

Node: University
Properties: docling_id, identifier

Node: Course
Properties: docling_id, enrolled_students, identifier, technologies

Node: Department
Properties: docling_id, identifier, professors

Node: Student
Properties: docling_id, identifier, skills

RELATIONSHIPS
(Department)-[:HAS_HEAD]->(Professor)
(Department)-[:HAS_STUDENT]->(Student)
(University)-[:HAS_DEPARTMENT]->(Department)
(University)-[:OFFERS_COURSE]->(Course)


# Prompt for NL->Cypher

In [14]:
cypher_prompt = (
    ChatPromptTemplate
    .from_messages(
        [
            (
                "system",
                """
                You are an expert Apache AGE Cypher
                query generator.

                Convert the user's question into
                exactly ONE read-only Cypher query.

                Use ONLY the provided graph schema.

                Rules:

                1. Use only node labels that exist
                in the schema.

                2. Use only relationship types that
                exist in the schema.

                3. Use only properties that exist
                in the schema.

                4. Never invent schema elements.

                5. Generate only the Cypher body.

                For example:

                MATCH (n)
                RETURN n

                Do NOT generate the PostgreSQL wrapper:

                SELECT *
                FROM cypher(...)

                6. Read-only operations only.

                Allowed:
                MATCH
                OPTIONAL MATCH
                WHERE
                WITH
                RETURN
                UNWIND
                ORDER BY
                SKIP
                LIMIT

                Forbidden:
                CREATE
                MERGE
                DELETE
                DETACH DELETE
                SET
                REMOVE
                DROP
                LOAD CSV

                7. Return only Cypher.

                8. Do not use markdown fences.

                9. Prefer returning scalar properties
                instead of complete vertices or edges,
                because the result will be consumed
                by an API.
                """
                            ),
                            (
                                "human",
                                """
                GRAPH SCHEMA:

                {graph_schema}

                QUESTION:

                {question}
                """
            ),
        ]
    )
)

# Build Chain


In [15]:
nl_to_cypher_chain = (
    cypher_prompt
    |
    llm
)

# Generate Cypher

In [16]:
def generate_cypher(
    question
):

    response = (
        nl_to_cypher_chain
        .invoke(
            {
                "question":
                    question,

                "graph_schema":
                    GRAPH_SCHEMA,
            }
        )
    )

    cypher = (
        response
        .content
        .strip()
    )

    cypher = re.sub(
        r"^```(?:cypher)?",
        "",
        cypher,
        flags=re.IGNORECASE,
    )

    cypher = re.sub(
        r"```$",
        "",
        cypher,
    )

    return cypher.strip()

# Test Cypher generation by Natural Langugage

In [17]:
question = (
    "Which students belong "
    "to each department?"
)

cypher_query = generate_cypher(
    question
)

print(cypher_query)

MATCH (d:Department)-[:HAS_STUDENT]->(s:Student)
RETURN d.identifier AS department, s.identifier AS student;


In [18]:
FORBIDDEN_CYPHER = [
    "CREATE",
    "MERGE",
    "DELETE",
    "DETACH",
    "SET",
    "REMOVE",
    "DROP",
    "LOAD CSV",
    "FOREACH",
]

In [19]:
def validate_read_only_cypher(
    cypher
):

    normalized = (
        cypher
        .upper()
        .strip()
    )

    for keyword in (
        FORBIDDEN_CYPHER
    ):

        if re.search(
            rf"\b{re.escape(keyword)}\b",
            normalized,
        ):

            raise ValueError(
                "Unsafe Cypher detected: "
                f"{keyword}"
            )

    return True

# Age Query Prompt

In [26]:
age_query_prompt = (
    ChatPromptTemplate
    .from_messages(
        [
            (
                "system",
                """
You generate read-only Apache AGE
Cypher queries.

Return JSON only.

Required format:

{{
  "cypher": "MATCH ... RETURN ...",
  "columns": [
    "column_1",
    "column_2"
  ]
}}

Rules:

- Use only the supplied schema.
- Never invent labels, properties,
  or relationships.
- Read-only queries only.
- Do not include the PostgreSQL
  SELECT FROM cypher wrapper.
- Every expression in RETURN must
  have an explicit alias.

Example Cypher:

MATCH (d:Department)-[:HAS_STUDENT]->(s:Student)
RETURN
    d.identifier AS department,
    s.identifier AS student

Then columns must be:

[
  "department",
  "student"
]

Return valid JSON only.
"""
            ),
            (
                "human",
                """
GRAPH SCHEMA:

{graph_schema}

QUESTION:

{question}
"""
            ),
        ]
    )
)

# Structure Age query generator

In [27]:
age_query_chain = (
    age_query_prompt
    |
    llm
)

In [28]:
def generate_age_query(
    question
):

    response = (
        age_query_chain.invoke(
            {
                "question":
                    question,

                "graph_schema":
                    GRAPH_SCHEMA,
            }
        )
    )

    text = (
        response
        .content
        .strip()
    )

    text = re.sub(
        r"^```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"```$",
        "",
        text,
    )

    result = json.loads(
        text.strip()
    )

    cypher = result["cypher"]
    columns = result["columns"]

    validate_read_only_cypher(
        cypher
    )

    return {
        "cypher": cypher,
        "columns": columns,
    }

In [29]:
def validate_column_name(
    name
):

    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        name,
    ):
        raise ValueError(
            f"Invalid column name: {name}"
        )

    return name

# Execute Age query

In [30]:
def execute_age_query(
    query_spec
):

    cypher = query_spec[
        "cypher"
    ]

    columns = query_spec[
        "columns"
    ]

    validate_read_only_cypher(
        cypher
    )

    validated_columns = [
        validate_column_name(column)
        for column in columns
    ]

    column_definition = ", ".join(
        f"{column} agtype"
        for column in validated_columns
    )

    sql = f"""
    SELECT *
    FROM cypher('{GRAPH_NAME}', $$
        {cypher}
    $$) AS (
        {column_definition}
    );
    """

    with conn.cursor() as cursor:

        cursor.execute(
            "LOAD 'age';"
        )

        cursor.execute(
            'SET search_path = '
            'ag_catalog, "$user", public;'
        )

        cursor.execute(sql)

        rows = cursor.fetchall()

    result = []

    for row in rows:

        result.append(
            {
                column:
                    normalize_agtype(value)

                for column, value
                in zip(
                    validated_columns,
                    row,
                )
            }
        )

    return result

# Test Age generation + execution

In [31]:
question = (
    "Which students belong "
    "to each department?"
)

query_spec = generate_age_query(
    question
)

print(
    "Generated Cypher:\n"
)

print(
    query_spec["cypher"]
)

print(
    "\nColumns:",
    query_spec["columns"]
)

Generated Cypher:

MATCH (d:Department)-[:HAS_STUDENT]->(s:Student)
RETURN d.identifier AS department, s.identifier AS student

Columns: ['department', 'student']


In [32]:
query_result = execute_age_query(
    query_spec
)

print(query_result)

[{'department': 'dept_cs', 'student': 'student_001'}, {'department': 'dept_cs', 'student': 'student_002'}]


In [33]:
answer_prompt = (
    ChatPromptTemplate
    .from_messages(
        [
            (
                "system",
                """
Answer the user's question using
only the supplied Apache AGE
query result.

Rules:

- Do not invent information.
- Do not use outside knowledge.
- If the result is empty, say that
  no matching data was found.
- Answer naturally and concisely.
"""
            ),
            (
                "human",
                """
QUESTION:

{question}

DATABASE RESULT:

{query_result}
"""
            ),
        ]
    )
)

In [34]:
answer_chain = (
    answer_prompt
    |
    llm
)

In [35]:
def generate_answer(
    question,
    query_result
):

    response = (
        answer_chain.invoke(
            {
                "question":
                    question,

                "query_result":
                    json.dumps(
                        query_result,
                        default=str,
                        indent=2,
                    ),
            }
        )
    )

    return (
        response
        .content
        .strip()
    )

In [36]:
def ask_age_graph(
    question,
    show_cypher=True,
    show_raw_result=False,
):

    query_spec = (
        generate_age_query(
            question
        )
    )

    query_result = (
        execute_age_query(
            query_spec
        )
    )

    answer = generate_answer(
        question=question,
        query_result=query_result,
    )

    if show_cypher:

        print(
            "\nGenerated Cypher:\n"
        )

        print(
            query_spec["cypher"]
        )


    if show_raw_result:

        print(
            "\nRaw AGE Result:\n"
        )

        print(query_result)


    return answer

# Final Flow for psql

In [37]:
answer = ask_age_graph(
    "Which students belong to each department?",
    show_cypher=True,
    show_raw_result=True,
)

print(
    "\nAnswer:\n"
)

print(answer)


Generated Cypher:

MATCH (d:Department)-[:HAS_STUDENT]->(s:Student)
RETURN d.identifier AS department, s.identifier AS student

Raw AGE Result:

[{'department': 'dept_cs', 'student': 'student_001'}, {'department': 'dept_cs', 'student': 'student_002'}]

Answer:

**dept_cs**: student_001, student_002
